In [ ]:
# Setup
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

from data import load_data
from model import (temporal_split, add_date_features, add_lag_features,
                   seasonal_naive_forecast, sarima_forecast,
                   train_lightgbm, lightgbm_forecast,
                   prophet_forecast, evaluate_forecast,
                   save_model, save_json, RANDOM_SEED)

from utils import save_fig, plot_forecast_vs_actual

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
ARTEFACTS_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_SEED)

In [ ]:
# 1. Load data + temporal split (last 8 weeks = test)
df = load_data()
df = add_date_features(df)
df = add_lag_features(df)

train, test = temporal_split(df, test_days=56)
print(f"Train: {len(train)} rows | date range: {train['Date'].min()} to {train['Date'].max()}")
print(f"Test:  {len(test)} rows | date range: {test['Date'].min()} to {test['Date'].max()}")

In [ ]:
# 2. Seasonal naive baseline
print("Fitting seasonal naive baseline...")
naive_pred = seasonal_naive_forecast(train, test)
naive_metrics = evaluate_forecast(test['Sales'], naive_pred)
print(f"  RMSPE = {naive_metrics['rmspe']:.2f}%  MAE = {naive_metrics['mae']:.0f}")

In [ ]:
# 3. SARIMA (slow — runs per-store; skip if sample is too large)
import time
print("Fitting SARIMA per-store (this can take a few minutes)...")
t0 = time.time()
try:
    sarima_pred = sarima_forecast(train, test)
    sarima_metrics = evaluate_forecast(test['Sales'], sarima_pred)
    print(f"  RMSPE = {sarima_metrics['rmspe']:.2f}%  MAE = {sarima_metrics['mae']:.0f}  (took {time.time()-t0:.1f}s)")
except Exception as e:
    print(f"  SARIMA failed: {e}")
    sarima_metrics = None

In [ ]:
# 4. LightGBM with lag features
print("Training LightGBM model...")
lgbm_model = train_lightgbm(train)
lgbm_pred = lightgbm_forecast(lgbm_model, train, test)
lgbm_metrics = evaluate_forecast(test['Sales'], lgbm_pred)
print(f"  RMSPE = {lgbm_metrics['rmspe']:.2f}%  MAE = {lgbm_metrics['mae']:.0f}")
save_model(lgbm_model, 'lightgbm_model', ARTEFACTS_DIR)

In [ ]:
# 5. Prophet (optional)
prophet_metrics = None
try:
    import prophet  # noqa: F401
    print("Prophet is installed — fitting per-store...")
    t0 = time.time()
    prophet_pred = prophet_forecast(train, test)
    prophet_metrics = evaluate_forecast(test['Sales'], prophet_pred)
    print(f"  RMSPE = {prophet_metrics['rmspe']:.2f}%  MAE = {prophet_metrics['mae']:.0f}  (took {time.time()-t0:.1f}s)")
except ImportError:
    print("Prophet not installed — skipping. Install with `pip install prophet` to enable this model.")

In [ ]:
# 6. Save all results
all_metrics = {
    "seasonal_naive": naive_metrics,
    "sarima":         sarima_metrics,
    "lightgbm":       lgbm_metrics,
    "prophet":        prophet_metrics,
}
# Filter out any None values
all_metrics = {k: v for k, v in all_metrics.items() if v is not None}
save_json(all_metrics, ARTEFACTS_DIR / 'forecast_results.json')
print("Saved →", ARTEFACTS_DIR / 'forecast_results.json')

# Print comparison table
import pandas as pd
print("\n=== Model comparison ===")
print(pd.DataFrame(all_metrics).T[['rmspe', 'mape', 'mae', 'rmse']].round(2))